# Experiment: Prior Work Implementation (GPU-MC Baseline vs GPU-MLMC)

Objective:
- Implement the prior-work baseline as single-level GPU Monte Carlo (SL-MC, no MLMC) on the same SDE and QoI.
- Compare baseline GPU-MC against GPU-MLMC (this work) on identical hardware/settings.

Done criteria:
- GPU-to-GPU table with epsilon, cost, runtime, CI half-width matching, runtime speedup, and cost ratio.
- Results saved to CSV for paper integration.


## Installation and Environment Check

This notebook assumes execution on an NVIDIA GPU machine (A6000 target) with CUDA and PyCUDA available.


In [ ]:
# Install/check dependencies
import importlib
import subprocess
import sys

core_packages = [
    'numpy',
    'pandas',
    'matplotlib',
    'networkx',
    'scipy',
]

for pkg in core_packages:
    module_name = pkg.replace('-', '_')
    try:
        importlib.import_module(module_name)
        print(f"OK: {pkg}")
    except Exception:
        print(f"Installing: {pkg}")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

# Check PyCUDA separately (do not auto-install by default)
try:
    import pycuda  # noqa: F401
    print('OK: pycuda')
except Exception as exc:
    print('WARN: pycuda is not importable in this runtime.')
    print(f'      details: {exc}')
    print('      On Colab, ensure GPU runtime is enabled and CUDA/PyCUDA are available.')

print('Dependency check complete.')


In [ ]:
# Imports and repo path setup + GPU environment metadata

from __future__ import annotations

import os
import subprocess
import sys
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd


def resolve_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / '.git').exists():
            return candidate
    return start


CWD = Path.cwd().resolve()
REPO_ROOT = resolve_repo_root(CWD)
SRC_DIR = REPO_ROOT / 'src'
DATASETS_PKG_DIR = REPO_ROOT / 'datasets'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
if str(DATASETS_PKG_DIR) not in sys.path:
    sys.path.insert(0, str(DATASETS_PKG_DIR))

from network.topology import TopologyGenerator, load_caida_topology
from network.traffic import PoissonTraffic
from gpu.parallel_mc import GPUMonteCarloSimulator, GPUMLMCSimulator, PYCUDA_AVAILABLE

try:
    from caida.loader import CAIDATopologyLoader
    CAIDA_LOADER_AVAILABLE = True
except Exception:
    CAIDA_LOADER_AVAILABLE = False
    CAIDATopologyLoader = None

print(f'Repo root: {REPO_ROOT}')
print(f'PYCUDA_AVAILABLE: {PYCUDA_AVAILABLE}')
print(f'CAIDA_LOADER_AVAILABLE: {CAIDA_LOADER_AVAILABLE}')

if not PYCUDA_AVAILABLE:
    raise RuntimeError(
        'PyCUDA/GPU backend not available. This notebook is intentionally GPU-to-GPU. '
        'Install CUDA+PyCUDA and rerun on the target machine.'
    )


def get_gpu_environment() -> dict:
    env = {}

    # PyCUDA device/driver info
    try:
        import pycuda.driver as cuda
        import pycuda.autoinit  # noqa: F401

        dev = cuda.Device(0)
        env['gpu_name'] = dev.name()
        env['gpu_total_memory_gb'] = round(dev.total_memory() / 1e9, 3)
        env['gpu_compute_capability'] = '.'.join(map(str, dev.compute_capability()))
        env['cuda_driver_version_raw'] = cuda.get_driver_version()
        env['cuda_runtime_version'] = '.'.join(map(str, cuda.get_version()))
    except Exception as exc:
        env['pycuda_probe_error'] = str(exc)

    # nvidia-smi info (driver + CUDA toolkit reported by driver)
    try:
        cmd = [
            'nvidia-smi',
            '--query-gpu=name,driver_version,cuda_version',
            '--format=csv,noheader,nounits',
        ]
        out = subprocess.check_output(cmd, text=True).strip().splitlines()
        if out:
            parts = [p.strip() for p in out[0].split(',')]
            if len(parts) >= 3:
                env['nvidia_smi_gpu_name'] = parts[0]
                env['nvidia_driver_version'] = parts[1]
                env['nvidia_cuda_version'] = parts[2]
    except Exception as exc:
        env['nvidia_smi_probe_error'] = str(exc)

    return env


gpu_env = get_gpu_environment()
print('GPU environment for reproducibility:')
display(pd.DataFrame([gpu_env]))


## Dataset Setup (Aligned with Implementation Notebook)

This notebook uses the same dataset style as your implementation notebook:
- Synthetic scenarios (generated)
- Real-world CAIDA AS topology (local file or loader download)
- SNAP/MAWI links registered for reproducibility


In [ ]:
# Dataset roots, links, and resolver utilities

DATASET_LINKS = {
    'snap_email_labels': 'https://snap.stanford.edu/data/email-Eu-core-department-labels.txt.gz',
    'snap_email_edges': 'https://snap.stanford.edu/data/email-Eu-core.txt.gz',
    'snap_ca_grqc': 'https://snap.stanford.edu/data/ca-GrQc.txt.gz',
    'mawi_202406191400': 'http://mawi.nezu.wide.ad.jp/mawi/samplepoint-F/2024/202406191400.pcap.gz',
}

DATASET_ROOTS = []
if Path('/notebooks').exists():
    DATASET_ROOTS.append(Path('/notebooks'))
    if (Path('/notebooks') / 'datasets').exists():
        DATASET_ROOTS.append(Path('/notebooks') / 'datasets')

DATASET_ROOTS.extend([
    REPO_ROOT / 'datasets',
    REPO_ROOT / 'datasets' / 'snap',
    REPO_ROOT / 'datasets' / 'caida',
    REPO_ROOT / 'datasets' / 'mawi',
])
DATASET_ROOTS = list(dict.fromkeys(DATASET_ROOTS))


def _find_existing_file(candidates):
    for p in candidates:
        if p.exists() and p.is_file():
            return p
    return None


def ensure_linked_dataset(url: str, filename: str, subdir: str, allow_download: bool = True):
    # Reuse existing local copy first
    candidates = []
    for root in DATASET_ROOTS:
        candidates.append(root / filename)
        candidates.append(root / subdir / filename)
    found = _find_existing_file(candidates)
    if found is not None:
        return found

    if not allow_download:
        return None

    # Download into repo datasets/<subdir>
    target_dir = REPO_ROOT / 'datasets' / subdir
    target_dir.mkdir(parents=True, exist_ok=True)
    target = target_dir / filename
    print(f'Downloading {filename} -> {target}')
    urllib.request.urlretrieve(url, target)
    return target


def find_caida_file(date: str = '20260101'):
    exact = f'{date}.as-rel2.txt.bz2'
    candidates = []
    for root in DATASET_ROOTS:
        candidates.append(root / exact)
        candidates.extend(root.glob('*.as-rel2.txt.bz2'))
        candidates.extend((root / 'caida').glob('*.as-rel2.txt.bz2'))
    return _find_existing_file(candidates)


# SNAP files are small enough to fetch if missing.
snap_and_small_files = {
    'email-Eu-core-department-labels.txt.gz': ensure_linked_dataset(
        DATASET_LINKS['snap_email_labels'],
        'email-Eu-core-department-labels.txt.gz',
        'snap',
        allow_download=True,
    ),
    'email-Eu-core.txt.gz': ensure_linked_dataset(
        DATASET_LINKS['snap_email_edges'],
        'email-Eu-core.txt.gz',
        'snap',
        allow_download=True,
    ),
    'ca-GrQc.txt.gz': ensure_linked_dataset(
        DATASET_LINKS['snap_ca_grqc'],
        'ca-GrQc.txt.gz',
        'snap',
        allow_download=True,
    ),
}

# MAWI trace is very large; keep local-only by default to avoid accidental 9GB+ download.
mawi_file = ensure_linked_dataset(
    DATASET_LINKS['mawi_202406191400'],
    '202406191400.pcap.gz',
    'mawi',
    allow_download=False,
)

dataset_rows = []
for name, p in snap_and_small_files.items():
    dataset_rows.append({'dataset': name, 'path': str(p) if p else 'MISSING', 'exists': bool(p and p.exists())})

dataset_rows.append({
    'dataset': '202406191400.pcap.gz (MAWI)',
    'path': str(mawi_file) if mawi_file else 'MISSING_LOCAL_ONLY',
    'exists': bool(mawi_file is not None),
    'note': 'Large file; not auto-downloaded by this notebook.'
})

caida_probe = find_caida_file('20260101')
dataset_rows.append({
    'dataset': 'CAIDA 20260101.as-rel2.txt.bz2',
    'path': str(caida_probe) if caida_probe is not None else 'NOT_FOUND_LOCAL',
    'exists': bool(caida_probe is not None),
})

print('Dataset availability (aligned with implementation notebook style):')
display(pd.DataFrame(dataset_rows))


## Baseline Definition, Fairness Rules, and Sanity Checks

Prior-work baseline in this notebook:
- Single-level GPU Monte Carlo (GPU-MC), no multilevel correction.

Comparison method:
- GPU-MLMC (this work).

Fairness controls enforced in code:
- Same QoI (`mean_queue`) for both methods.
- Same finest timestep `h_L` for each epsilon.
- Reproducible seed policy per `(network_size, epsilon)` scenario.
- Consistent cost definition:
  - GPU-MC cost: `N * (T/h_L)`
  - GPU-MLMC cost: `sum_l N_l * (T/h_l)`
- GPU warm-up calls are run once and excluded from timed measurements.

Reference framing used in paper text:
- GPU Monte Carlo practice: CUDA guide, GPU computing and reductions, GPU MC examples ([9]-[11], [24]).
- MLMC theory: Giles ([6], [7]).
- SDE discretization basis: Kloeden and Platen ([15]).


In [ ]:
# Experiment configuration and helper utilities
SEED = 42
QOI_METRIC = 'mean_queue'

# Requested epsilon grid
EPSILONS = [0.10, 0.05, 0.02]

# Synthetic scenarios from your implementation workflow
SYNTHETIC_NETWORK_SIZES = [100, 500]

# Real-world scenario (CAIDA AS topology)
RUN_CAIDA_SCENARIO = True
CAIDA_DATE = '20260101'
CAIDA_DOWNLOAD_IF_MISSING = True

# Shared simulator parameters
T_SIM = 10.0
BASE_DT = 0.2
REFINEMENT_FACTOR = 2
BIAS_CONSTANT = 1.0   # match GPUMLMCSimulator bias model: bias_estimate = sqrt(dt_finest)

# Runtime controls (increase for final A6000 runs if needed)
PILOT_SAMPLES_MC = 256
PILOT_SAMPLES_MLMC = 64
MAX_MC_SAMPLES = 200000
MIN_MC_SAMPLES = 2

# Strict equal-accuracy targeting for fair runtime comparison
CI_TARGET_FACTOR = 0.01
CI_MATCH_TOL = 0.15
MAX_CI_TUNE_ITERS = 6
EPSILON_TUNE_MIN_FACTOR = 0.25
EPSILON_TUNE_MAX_FACTOR = 2.00

# CUDA warm-up controls (excluded from measurement)
WARMUP_SAMPLES_MC = 256
WARMUP_PILOT_SAMPLES_MLMC = 16
WARMUP_EPSILON_MLMC = 0.20

OUTPUT_TABLE_DIR = REPO_ROOT / 'results' / 'results' / 'tables'
OUTPUT_TABLE_DIR.mkdir(parents=True, exist_ok=True)


def make_network(n_nodes: int, p: float = 0.15, seed: int = SEED):
    gen = TopologyGenerator(seed=seed)
    network = gen.generate_erdos_renyi(n_nodes=n_nodes, p=p, directed=False)
    network.set_link_properties(seed=seed)
    return network


def make_traffic(rate: float = 100.0, seed: int = SEED):
    return PoissonTraffic(rate=rate, seed=seed)


def scenario_seed(scenario_name: str, n_nodes: int, epsilon: float) -> int:
    # Reproducible seed per (scenario, network_size, epsilon)
    # Use deterministic hashing (not Python's randomized hash()).
    import hashlib

    eps_tag = int(round(1000.0 * epsilon))
    scenario_tag = int(hashlib.sha256(scenario_name.encode('utf-8')).hexdigest()[:8], 16) % 10000
    return int(SEED + 100000 + scenario_tag + 31 * n_nodes + eps_tag)


def choose_discretization_for_epsilon(
    epsilon: float,
    base_dt: float = BASE_DT,
    refinement_factor: int = REFINEMENT_FACTOR,
    bias_constant: float = BIAS_CONSTANT,
):
    # Bias model: bias ~ C * sqrt(dt). Split MSE budget equally across variance and bias terms.
    dt_target = (epsilon / (np.sqrt(2.0) * bias_constant)) ** 2
    dt_target = max(dt_target, 1e-6)

    raw_L = np.log(base_dt / dt_target) / np.log(refinement_factor)
    L_max = int(np.ceil(max(0.0, raw_L)))
    dt_finest = base_dt / (refinement_factor ** L_max)
    bias_est = bias_constant * np.sqrt(dt_finest)

    return {
        'L_max': L_max,
        'dt_finest': dt_finest,
        'dt_target': dt_target,
        'bias_estimate': bias_est,
    }


def finest_dt_from_L(L_max: int) -> float:
    return float(BASE_DT / (REFINEMENT_FACTOR ** L_max))


def ci_width(lower: float, upper: float) -> float:
    return float(upper - lower)


In [ ]:
# GPU-MC baseline and GPU-MLMC runner functions (with sanity checks)

WARMUP_DONE = {'mc': False, 'mlmc': False}


def warmup_gpu_mc(network, seed: int, dt_finest: float):
    sim = GPUMonteCarloSimulator(seed=seed)
    traffic = make_traffic(rate=100.0, seed=seed + 1)
    _ = sim.estimate(
        network=network,
        traffic=traffic,
        n_samples=WARMUP_SAMPLES_MC,
        T=T_SIM,
        dt=dt_finest,
        metric=QOI_METRIC,
        verbose=False,
    )


def warmup_gpu_mlmc(network, seed: int):
    sim = GPUMLMCSimulator(refinement_factor=REFINEMENT_FACTOR, seed=seed)
    traffic = make_traffic(rate=100.0, seed=seed + 1)
    _ = sim.mlmc_estimate_gpu(
        network=network,
        traffic=traffic,
        epsilon=WARMUP_EPSILON_MLMC,
        L_max=1,
        T=T_SIM,
        base_dt=BASE_DT,
        pilot_samples=WARMUP_PILOT_SAMPLES_MLMC,
        verbose=False,
    )


def load_caida_network(date: str, seed: int):
    caida_path = find_caida_file(date)

    if caida_path is None and CAIDA_DOWNLOAD_IF_MISSING and CAIDA_LOADER_AVAILABLE:
        try:
            loader = CAIDATopologyLoader(data_dir=REPO_ROOT / 'datasets' / 'caida')
            caida_path = loader.download_topology(date=date, force=False)
        except Exception as exc:
            print(f'WARN: Failed to download CAIDA {date}: {exc}')
            caida_path = None

    if caida_path is None:
        print(f'WARN: CAIDA {date} not available. Skipping real-world scenario.')
        return None, None

    network = load_caida_topology(caida_path, as_undirected=True, largest_component=True)
    network.set_link_properties(seed=seed)
    return network, str(caida_path)


def build_experiment_scenarios():
    scenarios = []

    for n_nodes in SYNTHETIC_NETWORK_SIZES:
        network = make_network(n_nodes=n_nodes, p=0.15, seed=SEED + n_nodes)
        scenarios.append({
            'scenario': f'synthetic_n{n_nodes}',
            'dataset_source': 'synthetic_generator',
            'network': network,
        })

    if RUN_CAIDA_SCENARIO:
        caida_network, caida_source = load_caida_network(CAIDA_DATE, seed=SEED + 777)
        if caida_network is not None:
            scenarios.append({
                'scenario': f'caida_{CAIDA_DATE}',
                'dataset_source': caida_source,
                'network': caida_network,
            })

    return scenarios


def run_gpu_mc_baseline(network, traffic, epsilon: float, dt_finest: float, seed: int):
    if not WARMUP_DONE['mc']:
        warmup_gpu_mc(network=network, seed=seed, dt_finest=dt_finest)
        WARMUP_DONE['mc'] = True

    gpu_mc = GPUMonteCarloSimulator(seed=seed)

    # Pilot variance at the same finest timestep used for this epsilon.
    pilot = gpu_mc.estimate(
        network=network,
        traffic=traffic,
        n_samples=PILOT_SAMPLES_MC,
        T=T_SIM,
        dt=dt_finest,
        metric=QOI_METRIC,
        verbose=False,
    )

    target_ci_half = CI_TARGET_FACTOR * float(epsilon)
    target_var_component = (target_ci_half / 1.96) ** 2
    n_required = int(np.ceil(max(1e-12, pilot.variance) / max(1e-12, target_var_component)))
    n_required = max(n_required, MIN_MC_SAMPLES)
    n_required = min(n_required, MAX_MC_SAMPLES)

    runtime_total = 0.0
    result = None
    for i in range(MAX_CI_TUNE_ITERS):
        result = gpu_mc.estimate(
            network=network,
            traffic=traffic,
            n_samples=n_required,
            T=T_SIM,
            dt=dt_finest,
            metric=QOI_METRIC,
            verbose=False,
        )
        runtime_total += float(result.metadata.get('gpu_time_seconds', 0.0))
        current_ci_half = 0.5 * ci_width(result.ci_lower, result.ci_upper)
        ratio = current_ci_half / max(target_ci_half, 1e-12)
        if abs(ratio - 1.0) <= CI_MATCH_TOL:
            break
        scale = ratio ** 2
        if ratio >= 1.0:
            n_new = int(np.ceil(n_required * scale))
        else:
            n_new = int(np.floor(n_required * scale))
        n_new = int(np.clip(n_new, MIN_MC_SAMPLES, MAX_MC_SAMPLES))
        if n_new == n_required and ratio < 1.0 and n_required > MIN_MC_SAMPLES:
            n_new = n_required - 1
        if n_new == n_required:
            break
        n_required = n_new

    runtime = float(runtime_total)
    throughput = float(result.n_samples / runtime) if runtime > 0 else np.nan

    # Error proxy (not true MSE): ci_half^2 + h_L.
    bias_sq = (BIAS_CONSTANT * np.sqrt(dt_finest)) ** 2
    error_proxy_ci2_plus_hL = (result.variance / result.n_samples) + bias_sq

    # Cost definition sanity check: N * (T/h_L)
    n_timesteps = int(T_SIM / dt_finest)
    cost_formula = float(result.n_samples * n_timesteps)
    cost_impl = float(result.computational_cost)
    cost_consistent = bool(np.isclose(cost_formula, cost_impl))

    return {
        'method': 'GPU-MC (Prior Work Baseline)',
        'qoi': QOI_METRIC,
        'seed': int(seed),
        'n_samples': int(result.n_samples),
        'dt': float(dt_finest),
        'estimate': float(result.mean),
        'variance': float(result.variance),
        'ci_lower': float(result.ci_lower),
        'ci_upper': float(result.ci_upper),
        'ci_width': ci_width(result.ci_lower, result.ci_upper),
        'ci_half': 0.5 * ci_width(result.ci_lower, result.ci_upper),
        'ci_target_half': float(target_ci_half),
        'ci_target_matched': bool(
            abs((0.5 * ci_width(result.ci_lower, result.ci_upper)) / max(target_ci_half, 1e-12) - 1.0) <= CI_MATCH_TOL
        ),
        'p95': float(np.percentile(result.samples, 95)),
        'p99': float(np.percentile(result.samples, 99)),
        'runtime_s': runtime,
        'throughput': throughput,
        'cost': cost_impl,
        'cost_formula': cost_formula,
        'cost_consistent': cost_consistent,
        'error_proxy_ci2_plus_hL': float(error_proxy_ci2_plus_hL),
    }


def run_gpu_mlmc(network, traffic, epsilon: float, L_max: int, seed: int):
    if not WARMUP_DONE['mlmc']:
        warmup_gpu_mlmc(network=network, seed=seed)
        WARMUP_DONE['mlmc'] = True

    gpu_mlmc = GPUMLMCSimulator(refinement_factor=REFINEMENT_FACTOR, seed=seed)

    target_ci_half = CI_TARGET_FACTOR * float(epsilon)
    eps_inner = float(epsilon)
    runtime_total = 0.0
    result = None

    for _ in range(MAX_CI_TUNE_ITERS):
        result = gpu_mlmc.mlmc_estimate_gpu(
            network=network,
            traffic=traffic,
            epsilon=eps_inner,
            L_max=L_max,
            T=T_SIM,
            base_dt=BASE_DT,
            pilot_samples=PILOT_SAMPLES_MLMC,
            verbose=False,
        )
        runtime_total += float(result.metadata.get('gpu_time_seconds', 0.0))
        current_ci_half = 0.5 * ci_width(result.ci_lower, result.ci_upper)
        ratio = current_ci_half / max(target_ci_half, 1e-12)
        if abs(ratio - 1.0) <= CI_MATCH_TOL:
            break
        eps_new = float(np.clip(
            eps_inner / max(ratio, 1e-12),
            EPSILON_TUNE_MIN_FACTOR * float(epsilon),
            EPSILON_TUNE_MAX_FACTOR * float(epsilon),
        ))
        if np.isclose(eps_new, eps_inner):
            break
        eps_inner = eps_new

    runtime = float(runtime_total)
    total_paths = int(sum(level.n_samples for level in result.level_stats))

    # Cost definition sanity check: sum_l N_l * (T/h_l)
    cost_formula = float(sum(level.n_samples * (T_SIM / level.dt) for level in result.level_stats))
    cost_impl = float(result.total_cost)
    cost_consistent = bool(np.isclose(cost_formula, cost_impl))

    finest_dt = float(min(level.dt for level in result.level_stats))

    return {
        'method': 'GPU-MLMC (This Work)',
        'qoi': QOI_METRIC,
        'seed': int(seed),
        'L_max': int(result.L_max),
        'finest_dt': finest_dt,
        'total_paths': total_paths,
        'estimate': float(result.estimate),
        'variance': float(result.variance),
        'ci_lower': float(result.ci_lower),
        'ci_upper': float(result.ci_upper),
        'ci_width': ci_width(result.ci_lower, result.ci_upper),
        'ci_half': 0.5 * ci_width(result.ci_lower, result.ci_upper),
        'ci_target_half': float(target_ci_half),
        'ci_target_matched': bool(
            abs((0.5 * ci_width(result.ci_lower, result.ci_upper)) / max(target_ci_half, 1e-12) - 1.0) <= CI_MATCH_TOL
        ),
        'epsilon_inner_used': float(eps_inner),
        'runtime_s': runtime,
        'cost': cost_impl,
        'cost_formula': cost_formula,
        'cost_consistent': cost_consistent,
        'mse': float(result.mse),
        'level_samples': [int(level.n_samples) for level in result.level_stats],
        'level_dts': [float(level.dt) for level in result.level_stats],
    }


def run_gpu_to_gpu_grid(epsilons=EPSILONS):
    rows = []
    scenarios = build_experiment_scenarios()

    if not scenarios:
        raise RuntimeError('No scenarios available to run.')

    for s in scenarios:
        scenario_name = s['scenario']
        dataset_source = s['dataset_source']
        network = s['network']

        for eps in epsilons:
            run_seed = scenario_seed(scenario_name=scenario_name, n_nodes=network.n_nodes, epsilon=eps)

            # Independent traffic instances but same seed policy for fairness/reproducibility
            traffic_mc = make_traffic(rate=100.0, seed=run_seed)
            traffic_mlmc = make_traffic(rate=100.0, seed=run_seed)

            disc = choose_discretization_for_epsilon(eps)
            dt_finest = float(disc['dt_finest'])
            L_max = int(disc['L_max'])

            mc = run_gpu_mc_baseline(network, traffic_mc, eps, dt_finest, seed=run_seed)
            mlmc = run_gpu_mlmc(network, traffic_mlmc, eps, L_max, seed=run_seed)

            runtime_speedup = mc['runtime_s'] / mlmc['runtime_s'] if mlmc['runtime_s'] > 0 else np.nan
            cost_ratio = mc['cost'] / mlmc['cost'] if mlmc['cost'] > 0 else np.nan

            # Must-have sanity checks
            same_qoi = (mc['qoi'] == mlmc['qoi'] == QOI_METRIC)
            same_finest_dt_hL = bool(np.isclose(mc['dt'], dt_finest) and np.isclose(mlmc['finest_dt'], finest_dt_from_L(L_max)))
            same_seed_policy = (mc['seed'] == mlmc['seed'] == run_seed)
            cost_definition_consistent = bool(mc['cost_consistent'] and mlmc['cost_consistent'])

            rows.append({
                'scenario': scenario_name,
                'dataset_source': dataset_source,
                'network_nodes': int(network.n_nodes),
                'network_edges': int(network.n_edges),
                'epsilon': float(eps),
                'run_seed': int(run_seed),
                'L_max': int(L_max),
                'dt_finest': float(dt_finest),
                'gpu_mc_cost': float(mc['cost']),
                'gpu_mlmc_cost': float(mlmc['cost']),
                'gpu_mc_cost_formula': float(mc['cost_formula']),
                'gpu_mlmc_cost_formula': float(mlmc['cost_formula']),
                'gpu_mc_runtime_s': float(mc['runtime_s']),
                'gpu_mlmc_runtime_s': float(mlmc['runtime_s']),
                'gpu_mc_error_proxy_ci2_plus_hL': float(mc['error_proxy_ci2_plus_hL']),
                'gpu_mlmc_internal_mse_estimate': float(mlmc['mse']),
                'gpu_mc_ci_width': float(mc['ci_width']),
                'gpu_mlmc_ci_width': float(mlmc['ci_width']),
                'gpu_mc_ci_half': float(mc['ci_half']),
                'gpu_mlmc_ci_half': float(mlmc['ci_half']),
                'ci_target_half': float(CI_TARGET_FACTOR * eps),
                'equal_accuracy_ci_targeted': bool(mc['ci_target_matched'] and mlmc['ci_target_matched']),
                'gpu_mc_p95': float(mc['p95']),
                'gpu_mc_p99': float(mc['p99']),
                'runtime_speedup': float(runtime_speedup),
                'cost_ratio': float(cost_ratio),
                'gpu_mc_samples': int(mc['n_samples']),
                'gpu_mlmc_total_paths': int(mlmc['total_paths']),
                'gpu_mlmc_level_samples': str(mlmc['level_samples']),
                'gpu_mlmc_level_dts': str(mlmc['level_dts']),
                'same_qoi': bool(same_qoi),
                'same_finest_dt_hL': bool(same_finest_dt_hL),
                'same_seed_policy': bool(same_seed_policy),
                'cost_definition_consistent': bool(cost_definition_consistent),
            })

            print(
                f"{scenario_name} eps={eps:.3f} | "
                f"MC {mc['runtime_s']:.3f}s vs MLMC {mlmc['runtime_s']:.3f}s | "
                f"speedup={runtime_speedup:.2f}x | checks: "
                f"qoi={same_qoi}, hL={same_finest_dt_hL}, seed={same_seed_policy}, cost={cost_definition_consistent}, "
                f"equal_ci={bool(mc['ci_target_matched'] and mlmc['ci_target_matched'])}"
            )

    return pd.DataFrame(rows)


In [ ]:
# Execute the GPU-to-GPU comparison grid
results_df = run_gpu_to_gpu_grid()
results_df = results_df.sort_values(['network_nodes', 'epsilon']).reset_index(drop=True)

print('Raw results:')
display(results_df)


In [ ]:
# Paper-ready comparison table requested by guide + sanity assertions

sanity_cols = [
    'scenario',
    'network_nodes',
    'epsilon',
    'same_qoi',
    'same_finest_dt_hL',
    'same_seed_policy',
    'cost_definition_consistent',
]

sanity_df = results_df[sanity_cols].copy()
print('Sanity checks (must all be True):')
display(sanity_df)

assert bool(sanity_df['same_qoi'].all()), 'QoI mismatch between GPU-MC and GPU-MLMC runs.'
assert bool(sanity_df['same_finest_dt_hL'].all()), 'Finest timestep h_L mismatch for at least one scenario.'
assert bool(sanity_df['same_seed_policy'].all()), 'Seed policy mismatch for at least one scenario.'
assert bool(sanity_df['cost_definition_consistent'].all()), 'Cost definition mismatch for at least one scenario.'

paper_table = results_df[
    [
        'scenario',
        'network_nodes',
        'epsilon',
        'gpu_mc_cost',
        'gpu_mlmc_cost',
        'gpu_mc_runtime_s',
        'gpu_mlmc_runtime_s',
        'gpu_mc_error_proxy_ci2_plus_hL',
        'gpu_mlmc_internal_mse_estimate',
        'gpu_mc_ci_width',
        'gpu_mlmc_ci_width',
        'runtime_speedup',
        'cost_ratio',
    ]
].copy()

paper_table.columns = [
    'Scenario',
    'Network Nodes',
    'epsilon',
    'GPU-MC Cost',
    'GPU-MLMC Cost',
    'GPU-MC Runtime (s)',
    'GPU-MLMC Runtime (s)',
    'GPU-MC MSE Proxy',
    'GPU-MLMC MSE',
    'GPU-MC CI Width',
    'GPU-MLMC CI Width',
    'Speedup (Runtime)',
    'Cost Ratio',
]

csv_path = OUTPUT_TABLE_DIR / 'prior_work_gpu_mc_vs_gpu_mlmc.csv'
sanity_csv_path = OUTPUT_TABLE_DIR / 'prior_work_gpu_mc_vs_gpu_mlmc_sanity.csv'

paper_table.to_csv(csv_path, index=False)
sanity_df.to_csv(sanity_csv_path, index=False)

print(f'Saved comparison table: {csv_path}')
print(f'Saved sanity table: {sanity_csv_path}')
display(paper_table)


In [ ]:
# Reviewer-facing visualization suite (saved for paper artifacts)
import ast
import matplotlib.pyplot as plt

plots_dir = OUTPUT_TABLE_DIR
plots_dir.mkdir(parents=True, exist_ok=True)

viz_df = results_df.copy().sort_values(['scenario', 'epsilon']).reset_index(drop=True)


def _save_fig(fig, filename: str):
    out = plots_dir / filename
    fig.savefig(out, dpi=180, bbox_inches='tight')
    print(f'Saved figure: {out}')


# 1) Log-log cost vs epsilon + empirical slope estimation
fig, ax = plt.subplots(figsize=(7.2, 5.2))
for scenario, sub in viz_df.groupby('scenario'):
    sub = sub.sort_values('epsilon')
    ax.plot(sub['epsilon'], sub['gpu_mc_cost'], marker='o', linewidth=1.6, label=f'{scenario} | GPU-MC')
    ax.plot(sub['epsilon'], sub['gpu_mlmc_cost'], marker='s', linewidth=1.6, linestyle='--', label=f'{scenario} | GPU-MLMC')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('RMS error target (epsilon)')
ax.set_ylabel('Total cost proxy (timesteps x paths)')
ax.set_title('Log-Log Cost vs Error (GPU-MC vs GPU-MLMC)')
ax.grid(True, which='both', alpha=0.3)
ax.legend(fontsize=8, ncol=2)

slope_rows = []
for scenario, sub in viz_df.groupby('scenario'):
    sub = sub.sort_values('epsilon')
    if len(sub) >= 2:
        x = np.log(sub['epsilon'].to_numpy(dtype=float))

        y_mc = np.log(sub['gpu_mc_cost'].to_numpy(dtype=float))
        m_mc, b_mc = np.polyfit(x, y_mc, 1)
        slope_rows.append({'scenario': scenario, 'method': 'GPU-MC', 'slope': float(m_mc), 'intercept': float(b_mc)})

        y_mlmc = np.log(sub['gpu_mlmc_cost'].to_numpy(dtype=float))
        m_mlmc, b_mlmc = np.polyfit(x, y_mlmc, 1)
        slope_rows.append({'scenario': scenario, 'method': 'GPU-MLMC', 'slope': float(m_mlmc), 'intercept': float(b_mlmc)})

_save_fig(fig, 'loglog_cost_vs_epsilon.png')
plt.show()

slopes_df = pd.DataFrame(slope_rows)
slopes_csv = plots_dir / 'empirical_complexity_slopes.csv'
slopes_df.to_csv(slopes_csv, index=False)
print(f'Saved slope table: {slopes_csv}')
display(slopes_df)

# 1b) Empirical slope bar chart
if not slopes_df.empty:
    pivot = slopes_df.pivot(index='scenario', columns='method', values='slope').sort_index()
    x = np.arange(len(pivot), dtype=float)
    width = 0.36

    fig, ax = plt.subplots(figsize=(7.8, 5.2))
    if 'GPU-MC' in pivot.columns:
        ax.bar(x - width / 2, pivot['GPU-MC'].to_numpy(dtype=float), width=width, label='GPU-MC')
    if 'GPU-MLMC' in pivot.columns:
        ax.bar(x + width / 2, pivot['GPU-MLMC'].to_numpy(dtype=float), width=width, label='GPU-MLMC')

    ax.axhline(-3.0, color='#666666', linestyle=':', linewidth=1.2, label='Theory ~ -3 (MC)')
    ax.axhline(-2.0, color='#999999', linestyle='--', linewidth=1.2, label='Theory ~ -2 (MLMC)')
    ax.set_ylabel('Empirical slope of log(cost) vs log(epsilon)')
    ax.set_title('Empirical Complexity Slopes by Scenario')
    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index.tolist(), rotation=20, ha='right')
    ax.grid(True, axis='y', alpha=0.3)
    ax.legend(fontsize=8, ncol=2)

    _save_fig(fig, 'empirical_slope_bars.png')
    plt.show()


# 2) Runtime vs epsilon (crossover behavior)
fig, ax = plt.subplots(figsize=(7.2, 5.2))
for scenario, sub in viz_df.groupby('scenario'):
    sub = sub.sort_values('epsilon')
    ax.plot(sub['epsilon'], sub['gpu_mc_runtime_s'], marker='o', linewidth=1.8, label=f'{scenario} | GPU-MC')
    ax.plot(sub['epsilon'], sub['gpu_mlmc_runtime_s'], marker='s', linewidth=1.8, linestyle='--', label=f'{scenario} | GPU-MLMC')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('RMS error target (epsilon)')
ax.set_ylabel('Runtime (seconds)')
ax.set_title('Runtime vs Error Target (GPU-MC vs GPU-MLMC)')
ax.grid(True, which='both', alpha=0.3)
ax.legend(fontsize=8, ncol=2)
_save_fig(fig, 'runtime_vs_epsilon.png')
plt.show()


# 3) Cost ratio vs epsilon
fig, ax = plt.subplots(figsize=(7.2, 5.2))
for scenario, sub in viz_df.groupby('scenario'):
    sub = sub.sort_values('epsilon')
    ax.plot(sub['epsilon'], sub['cost_ratio'], marker='o', linewidth=2.0, label=scenario)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('RMS error target (epsilon)')
ax.set_ylabel('Cost ratio (GPU-MC / GPU-MLMC)')
ax.set_title('Cost Ratio Growth at Tighter Accuracy')
ax.grid(True, which='both', alpha=0.3)
ax.legend(fontsize=8)
_save_fig(fig, 'cost_ratio_vs_epsilon.png')
plt.show()


# 4) MLMC level work allocation (tightest-epsilon, largest-network row)
fig, ax = plt.subplots(figsize=(7.2, 4.8))
level_row = viz_df.sort_values(['epsilon', 'network_nodes'], ascending=[True, False]).iloc[0]
level_samples = []
if 'gpu_mlmc_level_samples' in level_row.index:
    try:
        level_samples = ast.literal_eval(str(level_row['gpu_mlmc_level_samples']))
    except Exception:
        level_samples = []

if level_samples:
    levels = list(range(len(level_samples)))
    ax.bar(levels, level_samples, color='#1f77b4', alpha=0.9)
    ax.set_xlabel('MLMC level index l')
    ax.set_ylabel('Allocated samples N_l')
    ax.set_title(f"MLMC level allocation | {level_row['scenario']} | epsilon={level_row['epsilon']}")
    ax.grid(True, axis='y', alpha=0.3)
else:
    ax.text(0.5, 0.5, 'Level sample allocation not available in this run.', ha='center', va='center')
    ax.set_axis_off()

_save_fig(fig, 'mlmc_level_allocation.png')
plt.show()


# 5) Scaling view fixed: categorical scenario axis at tightest epsilon
tight_eps = float(viz_df['epsilon'].min())
scale_df = viz_df[np.isclose(viz_df['epsilon'], tight_eps)].sort_values(['network_nodes', 'scenario']).reset_index(drop=True)

x = np.arange(len(scale_df), dtype=float)
x_labels = [f"{s}\n(n={int(n)})" for s, n in zip(scale_df['scenario'], scale_df['network_nodes'])]

fig, ax = plt.subplots(figsize=(7.8, 4.8))
ax.plot(x, scale_df['gpu_mc_runtime_s'], marker='o', linewidth=2.0, label='GPU-MC')
ax.plot(x, scale_df['gpu_mlmc_runtime_s'], marker='s', linewidth=2.0, linestyle='--', label='GPU-MLMC')
ax.set_xticks(x)
ax.set_xticklabels(x_labels, rotation=20, ha='right')
ax.set_xlabel('Scenario at tightest epsilon')
ax.set_ylabel('Runtime (seconds)')
ax.set_title(f'Runtime scaling at epsilon={tight_eps}')
ax.grid(True, alpha=0.3)
ax.legend()
_save_fig(fig, 'runtime_scaling_tightest_epsilon.png')
plt.show()

print('Visualization artifact directory:')
print(plots_dir)


## Notes for Paper Wording

Suggested baseline statement:
- "We compare against a single-level GPU Monte Carlo baseline following standard GPU Monte Carlo practice [9-11,24], and evaluate the incremental benefit of MLMC variance reduction [6,7] under identical SDE discretization [15]."

Interpretation guidance:
- This notebook is intentionally GPU-to-GPU.
- The contribution is not that GPU is faster than CPU; it is that MLMC reduces work and runtime even on the same GPU baseline.
